# Laboratory Task 3 (01_Understanding Deep Learning)

<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Task 3</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Perform a forward and backward propagation in python using the inputs from <b>Laboratory Task 2</b>
</p>

```python
x = np.array([1, 0, 1])
y = np.array([1])

# use relu as the activation function.

# learning rate
lr = 0.001
```
</div>

## 1. Initialization and Helper Functions

This section sets up the initial matrices for inputs, targets, weights, and biases. It defines the learning rate (0.001), which controls how aggressively the network updates its parameters. It also defines the mathematical functions for the ReLU activation and its derivative, which is strictly required for computing gradients during backpropagation.

In [1]:
import numpy as np

# 1. Initialize Inputs and Targets
x = np.array([[1, 0, 1]])  
y = np.array([[1]])        
lr = 0.001

# Initialize Weights and Biases
W_h = np.array([[ 0.2, -0.3],
                [ 0.4,  0.1],
                [-0.5,  0.2]])

W_o = np.array([[-0.3],
                [-0.2]])

theta_h = np.array([[-0.4, 0.2]])
theta_o = np.array([[0.1]])

# Define ReLU and its derivative
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

## 2. Forward Propagation

The forward pass translates the input data into a prediction. It calculates the dot product of the inputs and hidden weights, adds the hidden biases, and passes the result through the ReLU activation. The resulting hidden activations (A_h) are then multiplied by the output weights and added to the output bias to generate the network's final prediction (A_o or y_hat).

In [2]:
# --- FORWARD PROPAGATION ---
# Hidden Layer
Z_h = np.dot(x, W_h) + theta_h
A_h = relu(Z_h)

# Output Layer
Z_o = np.dot(A_h, W_o) + theta_o
A_o = relu(Z_o)  # y_hat

## 3. Backward Propagation (Error & Gradient Calculation)

This block computes the error and propagates it backward to figure out how much each parameter contributed to that error. The output error (delta_o) is simply the difference between the prediction and the target. The hidden error (delta_h) is calculated by multiplying the output error by the output weights and the derivative of the hidden layer's ReLU function. The script then computes the exact gradients (the magnitude and direction of the required changes) for both weight matrices and bias arrays.   

In [3]:
# --- BACKWARD PROPAGATION ---
# 1. Output Error (delta_o)
delta_o = A_o - y

# 2. Hidden Layer Error (delta_h)
# Formula: (delta_o * W_o.T) * relu_derivative(Z_h)
delta_h = np.dot(delta_o, W_o.T) * relu_derivative(Z_h)

# 3. Calculate Gradients
dW_o = np.dot(A_h.T, delta_o)
dW_h = np.dot(x.T, delta_h)

dtheta_o = np.sum(delta_o, axis=0, keepdims=True)
dtheta_h = np.sum(delta_h, axis=0, keepdims=True)

## 4. Parameter Optimization

The final step applies the calculated gradients to update the network's parameters. By subtracting the product of the learning rate and the gradients from the original weights and biases, the network takes a small mathematical "step" in the direction that will minimize the error on the next forward pass.

In [4]:
 # 4. Update Weights and Biases
W_o = W_o - (lr * dW_o)
W_h = W_h - (lr * dW_h)
theta_o = theta_o - (lr * dtheta_o)
theta_h = theta_h - (lr * dtheta_h)

# Outputs
print("Updated Hidden Layer Weights (W_h):\n", W_h)
print("\nUpdated Output Layer Weights (W_o):\n", W_o)
print("\nUpdated Hidden Biases (theta_h):", theta_h)
print("\nUpdated Output Bias (theta_o):", theta_o)

Updated Hidden Layer Weights (W_h):
 [[ 0.2      -0.300184]
 [ 0.4       0.1     ]
 [-0.5       0.199816]]

Updated Output Layer Weights (W_o):
 [[-0.3     ]
 [-0.199908]]

Updated Hidden Biases (theta_h): [[-0.4       0.199816]]

Updated Output Bias (theta_o): [[0.10092]]


**Dimensionality Formatting for Backpropagation (x and y)**
Unlike the forward pass script where x and y were flat 1D arrays, they are explicitly initialized as 2D arrays here ([[1, 0, 1]] and [[1]]). This is a critical data setup requirement for backpropagation. Computing the weight gradients (e.g., np.dot(x.T, delta_h)) requires strict matrix transposition and alignment to distribute the error proportionally across the dimensions, which 1D arrays do not naturally support in NumPy.   

**The Learning Rate Throttle (lr = 0.001)**
The learning rate acts as the braking system for the network's parameter updates. Because the calculated gradients can sometimes be large and volatile, a very small learning rate like 0.001 scales them down. This ensures the network takes tiny, conservative steps during gradient descent, preventing the updates from wildly overshooting the optimal values and causing the loss to explode.

**Derivative Data Scaffolding (relu_derivative)**
Backpropagation fundamentally relies on the chain rule of calculus to assign "blame" for the final error to earlier layers. The relu_derivative setup is essential data infrastructure: it acts as a boolean gate ($z > 0$) that tells the network exactly which neurons were "active" during the forward pass so that error updates are only routed to the weights responsible for creating the output.   

**Hidden Layer Weights Update (Dying ReLU Effect)**
The output reveals that the first column of the hidden layer weights ([0.2, 0.4, -0.5]) remained entirely unchanged after backpropagation. During the forward pass, the first hidden node's pre-activation sum was negative, causing the ReLU function to output 0.0. Because the derivative of ReLU for any value less than or equal to 0 is strictly 0, the error gradient for this specific node was multiplied by zero. This completely halted the backward flow of information through that node, preventing its connected weights from receiving any updates.Output Layer Weights Adaptation (W_o)The output weights updated from [-0.3, -0.2] to [-0.3, -0.199908]. The first weight remained -0.3 because the first hidden node sent a signal of 0.0, resulting in a gradient of zero for that connection. The second weight updated because it received an active signal (0.1) from the second hidden node. To counter the severe negative error (-0.92), the optimizer increased this weight slightly from -0.2 to -0.199908, effectively instructing the network to pull the final prediction higher on the next pass.Bias Vector Corrections (theta_o and theta_h)The output bias (theta_o) experienced the most direct correction, updating from 0.1 to 0.10092. Because the bias gradient is essentially a direct pass-through of the output error multiplied by the learning rate ($-0.92 \times 0.001 = -0.00092$), the network shifted the entire output activation function upwards by subtracting this negative gradient. The hidden bias (theta_h) mirrored the hidden weights' behavior: the first node's bias remained frozen at -0.4 due to the zero-gradient from the inactive ReLU, while the second node's bias adjusted slightly downward to -0.199816 to fine-tune its active signal.